In [16]:
import pandas as pd
import re

# 1. Cargar archivo
df = pd.read_excel(r"C:\Users\SD\Downloads\Normativa nacional víctimas 1869-2023.xlsx")

# 2. Ver columnas
print("Columnas:", df.columns)

# 3. Limpieza de texto
def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    
    texto = str(texto).lower()
    
    # eliminar saltos de línea
    texto = texto.replace("\n", " ")
    
    # eliminar caracteres raros
    texto = re.sub(r"[^a-záéíóúñ0-9\s]", " ", texto)
    
    # eliminar espacios extra
    texto = re.sub(r"\s+", " ", texto).strip()
    
    return texto

# 4. Aplicar limpieza sobre la columna "Artículos"
df["texto_limpio"] = df["Artículos"].apply(limpiar_texto)

# 5. Crear dataset final
dataset = df[[
    "Tipo",
    "Número",
    "Título",
    "texto_limpio",
    "caso_ok"
]].copy()

# 6. Eliminar filas sin texto
dataset = dataset[dataset["texto_limpio"] != ""]

# 7. Agregar longitud del texto (útil para análisis)
dataset["longitud_texto"] = dataset["texto_limpio"].apply(len)

# 8. Resetear índice
dataset = dataset.reset_index(drop=True)

# 9. Guardar dataset limpio

dataset.to_excel(r"C:\Users\SD\Downloads\dataset_normas.xlsx", index=False)

# 10. Info básica
print("\nDataset creado:")
print(dataset.shape)
print(dataset.head())

Columnas: Index(['Tipo', 'Número', 'Añosanción', 'texto completo', 'periodos',
       'periodos2', 'rango', 'presidencia', 'decadas', 'Título', 'Subtítulo',
       'Resumen', 'Artículos', 'caso_ok', 'caso', 'internacional_ok', 'tema',
       'subtema', 'tema2', 'subtema2', 'pol_ok1', 'pol_ok2', 'Comentarios'],
      dtype='object')

Dataset creado:
(144, 6)
  Tipo Número                              Título  \
0  Ley    340                        CODIGO CIVIL   
1  Ley   2372  CODIGO PROCESAL PENAL DE LA NACION   
2  Ley   2637                  CODIGO DE COMERCIO   
3  Ley   4960                       GASTO PéBLICO   
4  Ley   4964                  EMERGENCIA PéBLICA   

                                        texto_limpio  caso_ok  longitud_texto  
0  articulo 1079 la obligación de reparar el daño...        0             976  
1  articulo 170 la persona particularmente ofendi...        0            1477  
2  art 184 en caso de muerte o lesión de un viaje...        0             372  
3

In [17]:
dataset

,Tipo,Número,Título,texto_limpio,caso_ok,longitud_texto
0,Ley,340,CODIGO CIVIL,articulo 1079 la obligación de reparar el daño...,0,976
1,Ley,2372,CODIGO PROCESAL PENAL DE LA NACION,articulo 170 la persona particularmente ofendi...,0,1477
2,Ley,2637,CODIGO DE COMERCIO,art 184 en caso de muerte o lesión de un viaje...,0,372
3,Ley,4960,GASTO PéBLICO,terremoto valparaíso agosto 1906,1,32
4,Ley,4964,EMERGENCIA PéBLICA,terremoto valparaíso agosto 1906,1,32
...,...,...,...,...,...,...
139,Ley,27547,LEY DE LA SOCIEDAD NACIONAL DE LA CRUZ ROJA,artículo 4 se autoriza a cruz roja argentina a...,0,1009
140,Decreto,459,DUELO NACIONAL,artículo 1 declárase duelo nacional en todo el...,1,226
141,Ley,27656,REPARACIÓN HISTÓRICA,artículo 1 dispónese la inscripción de la cond...,0,851
142,Ley,27696,PROGRAMA MÉDICO OBLIGATORIO DE LAS OBRAS SOCIA...,artículo 1 la presente ley tiene por objeto in...,0,1875


In [18]:
dataset.loc[10, "caso_ok"] = 1  # ejemplo

In [19]:
dataset = dataset.dropna(subset=["texto_limpio"])
dataset = dataset[dataset["texto_limpio"].str.len() > 30]

In [20]:
import json

ruta = r"C:\Users\SD\Downloads\dataset_normas.jsonl"

with open(ruta, "w", encoding="utf-8") as f:
    for _, row in dataset.iterrows():
        ejemplo = {
            "text": row["texto_limpio"],
            "caso_ok": int(row["caso_ok"])
        }
        json.dump(ejemplo, f, ensure_ascii=False)
        f.write("\n")

print("Dataset listo")

Dataset listo


In [29]:
print(dataset)
print(dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['text', 'caso_ok'],
        num_rows: 108
    })
    test: Dataset({
        features: ['text', 'caso_ok'],
        num_rows: 28
    })
})
['text', 'caso_ok']


In [30]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

# Cargar dataset
dataset = load_dataset("json", data_files=r"C:\Users\SD\Downloads\dataset_normas.jsonl")

# Dividir train/test
dataset = dataset["train"].train_test_split(test_size=0.2)

# Modelo (MULTILINGÜE → sirve para español)
model_name = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length")
    tokens["labels"] = example["caso_ok"]
    return tokens


dataset = dataset.map(tokenize)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir="./modelo_normas",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    logging_dir="./logs"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)

trainer.train()

Map:   0%|          | 0/108 [00:00<?, ? examples/s]

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:01<?, ?it/s]

TrainOutput(global_step=162, training_loss=0.271066806934498, metrics={'train_runtime': 2478.759, 'train_samples_per_second': 0.131, 'train_steps_per_second': 0.065, 'total_flos': 42919437164544.0, 'train_loss': 0.271066806934498, 'epoch': 3.0})

In [31]:
predictions = trainer.predict(dataset["test"])

C:\Users\SD\.anaconda\New folder\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [32]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("Accuracy:", accuracy_score(labels, preds))
print("Precision:", precision_score(labels, preds))
print("Recall:", recall_score(labels, preds))
print("F1:", f1_score(labels, preds))

Accuracy: 0.8571428571428571
Precision: 0.3333333333333333
Recall: 1.0
F1: 0.5


In [34]:
model.save_pretrained("./modelo_normas", safe_serialization=False)
tokenizer.save_pretrained("./modelo_normas")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SafetensorError: Error while serializing: I/O error: The requested operation cannot be performed on a file with a user-mapped section open. (os error 1224)

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification", model="./modelo_normas")

texto = "La ley garantiza asistencia y reparación a víctimas del delito"

print(classifier(texto))